## Part 1: Data

In [1]:
!pip install lxml
import pandas as pd
import requests
from io import StringIO

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

response = requests.get(url, headers=headers)
response.raise_for_status()

tables = pd.read_html(StringIO(response.text))  # <-- the fix
sp500 = tables[0]
tickers = sp500["Symbol"].tolist()
tickers = [t.replace(".", "-") for t in tickers]

print(len(tickers), tickers[:5])

503 ['MMM', 'AOS', 'ABT', 'ABBV', 'ACN']


In [2]:
!pip install yfinance --upgrade --no-cache-dir

In [3]:
import yfinance as yf 
data_2025 = yf.download(tickers, start='2025-01-01', end='2025-12-31', auto_adjust=False, progress=False)
data_2026 = yf.download(tickers, start='2026-01-01', end='2026-09-15', auto_adjust=False, progress=False)

$HONA: Data doesn't exist for startDate = 1735707600, endDate = 1767157200
$FDXF: Data doesn't exist for startDate = 1735707600, endDate = 1767157200

2 Failed downloads:
['HONA', 'FDXF']: Data doesn't exist for startDate = 1735707600, endDate = 1767157200


HONA (Honeywell Aerospace) was added to the S&P 500 on 2026-06-29, and FDXF (FedEx Freight) on 2026-06-01.
Both are recent corporate spinoffs. Since we need data starting 2025-01-01, neither stock existed yet under that ticker, hence "data doesn't exist" for the 2025 date range.

dropna will remove these two columns automatically since they'll be almost empty for 2025, and the intersection with 2026 columns will drop them from that side too, if needed.

In [4]:
adj_close_2025 = data_2025['Adj Close']
adj_close_2026 = data_2026['Adj Close']

adj_close_2025 = adj_close_2025.dropna(axis=1, thresh=int(0.95*len(adj_close_2025)))
adj_close_2026 = adj_close_2026.dropna(axis=1, thresh=int(0.95*len(adj_close_2026)))

common = adj_close_2025.columns.intersection(adj_close_2026.columns)
adj_close_2025 = adj_close_2025[common]
adj_close_2026 = adj_close_2026[common]

print(adj_close_2025.shape, adj_close_2026.shape)

(249, 499) (175, 499)


249 trading days for full-year 2025

175 trading days for 2026 through mid-September (roughly 9.5 months)

499 stocks survived the cleanup (503 minus HONA and FDXF, plus a couple more that likely got dropped by the dropna threshold)

In [ ]:
returns_2025 = adj_close_2025.pct_change().iloc[1:]
returns_2026 = adj_close_2026.pct_change().iloc[1:]

# Check for gaps
print("2025 missing values per column (nonzero only):")
print(returns_2025.isna().sum()[returns_2025.isna().sum() > 0])

print("2026 missing values per column (nonzero only):")
print(returns_2026.isna().sum()[returns_2026.isna().sum() > 0])

2025 missing values per column (nonzero only):
Ticker
FISV    2
dtype: int64
2026 missing values per column (nonzero only):
Series([], dtype: int64)


FISV (Fiserv) has 2 missing values in the 2025 data (probably a data gap from yfinance, or possibly because Fiserv appears twice in the constituent table under related tickers FI/FISV). Those 2 missing days in that column were enough to drop 2 entire days across all 499 stocks with the blanket dropna().

In [7]:
returns_2025 = returns_2025.dropna(axis=1)
returns_2026 = returns_2026.dropna(axis=1)

common = returns_2025.columns.intersection(returns_2026.columns)
returns_2025 = returns_2025[common]
returns_2026 = returns_2026[common]

returns_2025.to_csv('returns_2025.csv')
returns_2026.to_csv('returns_2026.csv')

print(returns_2025.shape, returns_2026.shape)

(248, 498) (174, 498)


The shapes are one row shorter than before in each case (248 and 174), since pct_change() can't compute a return for the very first day — there's no prior price to compare it to.
All 498 stocks now have complete, gap-free return series for both years, and no day was lost due to one bad column.

## Part 2:

> Build the LP that minimizes CVaR on the 2025 data at β = 0.95 with R = 0.02%/day, then evaluate that fixed portfolio's CVaR on the 2026 data.

In [8]:
returns_2025.mean().max()

np.float64(0.008320407764267172)

The best-performing single stock in 2025 data averages about 0.83% per day which is far above the R = 0.02% per day (0.0002) floor required. So, the return constraint is easily satisfiable. The LP should not be infeasible on that account.

In [1]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np

def min_cvar_portfolio_gurobi(Y, beta=0.95, R=0.0002):
    Y = np.asarray(Y)
    q, n = Y.shape
    ybar = Y.mean(axis=0)

    m = gp.Model("cvar")
    m.setParam('OutputFlag', 0)  # silence solver logs

    x = m.addVars(n, lb=0, name="x")             # weights, x >= 0
    alpha = m.addVar(lb=-GRB.INFINITY, name="alpha")  # free variable
    u = m.addVars(q, lb=0, name="u")              # overshoot, u >= 0

    # Objective: alpha + (1/((1-beta)*q)) * sum(u)
    m.setObjective(alpha + (1.0/((1-beta)*q)) * gp.quicksum(u[k] for k in range(q)), GRB.MINIMIZE)

    # u_k >= -y_k'x - alpha
    for k in range(q):
        m.addConstr(u[k] >= -gp.quicksum(Y[k,j]*x[j] for j in range(n)) - alpha)

    # Weights sum to 1
    m.addConstr(gp.quicksum(x[j] for j in range(n)) == 1)

    # Expected return floor
    m.addConstr(gp.quicksum(ybar[j]*x[j] for j in range(n)) >= R)

    m.optimize()

    if m.status != GRB.OPTIMAL:
        raise ValueError(f"Gurobi did not find optimal solution, status: {m.status}")

    weights = np.array([x[j].X for j in range(n)])
    return weights, alpha.X, m.objVal

In [7]:
import yfinance as yf

# S&P 500 index returns
spx = yf.download("^GSPC", start="2025-01-01", end="2026-09-15", auto_adjust=False)["Adj Close"]
spx_returns = spx.pct_change().dropna()

spx_2025 = spx_returns[spx_returns.index.year == 2025].to_numpy().reshape(-1, 1)
spx_2026 = spx_returns[spx_returns.index.year == 2026].to_numpy().reshape(-1, 1)
spx_returns.to_csv("spx_returns.csv")

[*********************100%***********************]  1 of 1 completed


In [8]:
import pandas as pd

# Load daily return data
df_2025 = pd.read_csv("returns_2025.csv", parse_dates=["Date"], index_col="Date")
df_2026 = pd.read_csv("returns_2026.csv", parse_dates=["Date"], index_col="Date")

tickers = df_2025.columns.tolist()      # pulled dynamically
Y_2025 = df_2025.to_numpy()
Y_2026 = df_2026[tickers].to_numpy()    # reindex to guarantee matching column order

In [9]:
# CVaR of a FIXED portfolio x, evaluated against return sample Y (out-of-sample use)
def evaluate_cvar_gurobi(x, Y, beta=0.95):
    x = np.asarray(x)
    Y = np.asarray(Y)
    q, n = Y.shape
    losses = -Y @ x  # portfolio loss each period, weights fixed

    m = gp.Model("cvar_eval")
    m.setParam('OutputFlag', 0)
    alpha = m.addVar(lb=-GRB.INFINITY, name="alpha")
    u = m.addVars(q, lb=0, name="u")
    m.setObjective(alpha + (1.0/((1-beta)*q)) * gp.quicksum(u[k] for k in range(q)), GRB.MINIMIZE)
    for k in range(q):
        m.addConstr(u[k] >= losses[k] - alpha)
    m.optimize()
    return alpha.X, m.objVal

In [10]:
# In-sample: optimal 2025 portfolio at beta=0.95, R=0.02%
x_2025, alpha_2025, cvar_2025_insample = min_cvar_portfolio_gurobi(Y_2025, beta=0.95, R=0.0002)

# Out-of-sample: same weights, evaluated on 2026 data
alpha_oos, cvar_2026_oos = evaluate_cvar_gurobi(x_2025, Y_2026, beta=0.95)

# Q2b: S&P index CVaR (a "portfolio" with a single asset, weight 1.0)
_, cvar_spx_2025 = evaluate_cvar_gurobi(np.array([1.0]), spx_2025, beta=0.95)
_, cvar_spx_2026 = evaluate_cvar_gurobi(np.array([1.0]), spx_2026, beta=0.95)

print(f"2025 in-sample CVaR (optimized):     {cvar_2025_insample:.6f}")
print(f"2026 out-of-sample CVaR (2025 x):     {cvar_2026_oos:.6f}")
print(f"S&P 500 CVaR 2025:                    {cvar_spx_2025:.6f}")
print(f"S&P 500 CVaR 2026:                    {cvar_spx_2026:.6f}")

2025 in-sample CVaR (optimized):     0.009329
2026 out-of-sample CVaR (2025 x):     0.011630
S&P 500 CVaR 2025:                    0.027217
S&P 500 CVaR 2026:                    0.017692


In-sample, the portfolio's daily CVaR (0.93%) is roughly a third of the S&P 500's CVaR over the same period (2.72%) — a large tail-risk reduction relative to just holding the index.

Performance degraded out-of-sample, as expected. CVaR rose from 0.93% to 1.16% (about a 25% increase) when the fixed 2025 weights were evaluated against 2026 data. This is in-sample overfitting: the LP found the weights that were optimal for the specific 248 return scenarios in 2025, and those scenarios don't repeat exactly in 2026.

The S&P's own CVaR actually fell from 2.72% (2025) to 1.77% (2026), implying that the broad market was calmer in 2026. Yet the portfolio's CVaR rose. 

Ratio of the portfolio's CVaR to the S&P's CVaR in each period is a rough measure of how much tail-risk reduction the optimization achieved relative to just holding the index.

- 2025: 0.009329 / 0.027217 ≈ 0.343 — portfolio's CVaR was about 34% of the S&P's
- 2026: 0.011630 / 0.017692 ≈ 0.657 — portfolio's CVaR was about 66% of the S&P's

So the edge over the index shrank a lot, from cutting tail risk by roughly two-thirds down to cutting it by only about a third. The ratio of portfolio risk to market risk went from about 0.34 (2025) to about 0.66 (2026) — the portfolio captured much less of its risk-reduction advantage out-of-sample, even though the overall market environment was more benign. Non-stationarity: the stock-level correlation and tail structure the optimizer exploited in 2025 partially broke down by 2026, even as aggregate volatility fell.

Should we stick with the same portfolio? No, not because the portfolio became bad (it still beat the S&P in 2026 in absolute terms), but because its edge shrank substantially, which is exactly the risk of optimizing tightly to one historical sample.

## Part 3: Different β values

In [11]:
betas = [0.90, 0.95, 0.99]
part3_results = {}

for b in betas:
    x_b, alpha_b, cvar_b = min_cvar_portfolio_gurobi(Y_2025, beta=b, R=0.0002)
    n_nonzero = int(np.sum(x_b > 1e-6))          # how concentrated vs. diversified
    hhi = float(np.sum(x_b**2))                  # Herfindahl index — lower = more diversified
    part3_results[b] = {"weights": x_b, "alpha": alpha_b, "cvar": cvar_b, "n_nonzero": n_nonzero, "hhi": hhi}
    print(f"beta={b}: CVaR={cvar_b:.6f}  alpha(VaR)={alpha_b:.6f}  "
          f"nonzero holdings={n_nonzero}  HHI={hhi:.4f}")

beta=0.9: CVaR=0.007463  alpha(VaR)=0.004298  nonzero holdings=31  HHI=0.0604
beta=0.95: CVaR=0.009329  alpha(VaR)=0.006171  nonzero holdings=31  HHI=0.0530
beta=0.99: CVaR=0.012342  alpha(VaR)=0.012342  nonzero holdings=17  HHI=0.1165


CVaR rises monotonically with β (0.75% → 0.93% → 1.23%) which is mechanically expected, since higher β means averaging over a deeper, more extreme slice of the loss tail, and extreme losses are larger by definition.

Concentration increases at β=0.99, not decreases: holdings drop from 31 to 17 names, and HHI nearly doubles (0.053 → 0.117). Probably because at β=0.99 with a 248-day sample, (1-β)·q ≈ 2.5, only the single worst day or two is actually driving the 'u' terms in the objective. With so few scenarios effectively in play, the LP has much less to average over and less incentive to hedge broadly across many names — it can concentrate in whatever handful of stocks had the best (or least bad) performance on those specific worst days, rather than spreading risk the way it does when dozens of moderately-bad days all matter (β=0.90/0.95).

alpha == CVaR exactly at β=0.99 is a direct consequence of that same small-sample effect — when there are only ~2 scenarios feeding u, the LP's VaR estimate (alpha) and CVaR estimate can converge, since there's almost no "excess beyond VaR" left to average once so few points qualify as tail scenarios.

Limitation: β=0.99 is sensitive to sample size here, and a larger daily sample (multiple years) would likely give a more stable, less degenerate answer at that confidence level.

HHI (Herfindahl-Hirschman Index) was added as a quick, single-number way to measure how concentrated vs. diversified a portfolio's weights are.

It's computed as the sum of squared weights: `HHI = Σ xⱼ²`
- If a portfolio is split evenly across `n` assets, HHI = 1/n. So 31 equal holdings would give HHI ≈ 0.032.
- If a portfolio is 100% in one asset, HHI = 1.0.
- Lower HHI → more spread out; higher HHI → more concentrated in a few names.

Part 3 asks "how do different β values affect the portfolio allocation?" and that question isn't really answerable from the CVaR number alone, since CVaR just tells you the risk level, not the shape of the portfolio that produced it. Two portfolios could have the same CVaR while one holds 5 stocks and another holds 50. `n_nonzero` (count of holdings) and HHI together let us characterize how the allocation changed across β=0.90/0.95/0.99 which let us catch that β=0.99 concentrated into fewer names (17 vs. 31) with a much higher HHI (0.117 vs. ~0.05), rather than just reporting "CVaR went up."

## Part 4 — Minimize the maximum monthly CVaR

Instead of one CVaR expression, we get 12 separate CVaR expressions (one per month), each with its own alpha_m and u_m variables, all sharing the same portfolio x. 
We minimize the largest of those 12 monthly CVaRs — a classic min-max linearization using an auxiliary variable Z that's forced to be ≥ every month's CVaR.

In [12]:
def min_max_monthly_cvar_gurobi(returns_df, beta=0.95, R=0.0002):
    """Minimize the WORST month's CVaR (not the average) over the given period."""
    Y = returns_df.to_numpy()
    dates = returns_df.index
    n = Y.shape[1]
    ybar = Y.mean(axis=0)
    months = sorted(dates.month.unique())

    m = gp.Model("minimax_monthly_cvar")
    m.setParam('OutputFlag', 0)

    x = m.addVars(n, lb=0, name="x")
    Z = m.addVar(lb=-GRB.INFINITY, name="Z")  # worst-month CVaR — this is what we minimize

    alpha, u = {}, {}
    for mo in months:
        idx = np.where(dates.month == mo)[0]
        q_m = len(idx)
        alpha[mo] = m.addVar(lb=-GRB.INFINITY, name=f"alpha_{mo}")
        u[mo] = m.addVars(q_m, lb=0, name=f"u_{mo}")

        cvar_m = alpha[mo] + (1.0/((1-beta)*q_m)) * gp.quicksum(u[mo][i] for i in range(q_m))
        m.addConstr(Z >= cvar_m)  # Z must dominate every month's CVaR

        for i, k in enumerate(idx):
            m.addConstr(u[mo][i] >= -gp.quicksum(Y[k, j]*x[j] for j in range(n)) - alpha[mo])

    m.setObjective(Z, GRB.MINIMIZE)
    m.addConstr(gp.quicksum(x[j] for j in range(n)) == 1)
    m.addConstr(gp.quicksum(ybar[j]*x[j] for j in range(n)) >= R)

    m.optimize()
    if m.status != GRB.OPTIMAL:
        raise ValueError(f"Gurobi did not find optimal solution, status: {m.status}")

    weights = np.array([x[j].X for j in range(n)])
    monthly_cvars = {mo: alpha[mo].X + (1.0/((1-beta)*len(np.where(dates.month==mo)[0])))
                      * sum(u[mo][i].X for i in range(len(np.where(dates.month==mo)[0])))
                      for mo in months}
    return weights, Z.X, monthly_cvars

In [13]:
x_minimax, worst_month_cvar, monthly_cvars_minimax = min_max_monthly_cvar_gurobi(df_2025, beta=0.95, R=0.0002)
_, avg_cvar_minimax = evaluate_cvar_gurobi(x_minimax, Y_2025, beta=0.95)

print(f"Part 2 (avg CVaR):      x_2025 -> avg CVaR = {cvar_2025_insample:.6f}")
print(f"Part 4 (minimax CVaR):  worst-month CVaR = {worst_month_cvar:.6f}, avg CVaR = {avg_cvar_minimax:.6f}")

Part 2 (avg CVaR):      x_2025 -> avg CVaR = 0.009329
Part 4 (minimax CVaR):  worst-month CVaR = 0.012274, avg CVaR = 0.012343


## Part 5 — Monthly reoptimization with a rolling 1-year window

In [14]:
# One continuous return series across both years
full_df = pd.concat([df_2025, df_2026]).sort_index()

def rolling_monthly_portfolios(full_df, target_months, beta=0.95, R=0.0002):
    """For each (year, month), train on the trailing 12 months, then evaluate out-of-sample
    on that target month's actual returns."""
    results = {}
    for target_year, target_month in target_months:
        window_end = pd.Timestamp(year=target_year, month=target_month, day=1) - pd.Timedelta(days=1)
        window_start = window_end - pd.DateOffset(years=1) + pd.Timedelta(days=1)
        Y_train = full_df.loc[window_start:window_end].to_numpy()

        x, _, cvar_in = min_cvar_portfolio_gurobi(Y_train, beta=beta, R=R)

        mask = (full_df.index.year == target_year) & (full_df.index.month == target_month)
        Y_target = full_df.loc[mask].to_numpy()
        _, cvar_oos = evaluate_cvar_gurobi(x, Y_target, beta=beta)

        results[(target_year, target_month)] = {"weights": x, "cvar_oos": cvar_oos, "n_days": len(Y_target)}
    return results

# 2026 months with data available (through Sept 14 — Sept is a partial month)
target_months = [(2026, m) for m in range(1, 10)]
monthly_results = rolling_monthly_portfolios(full_df, target_months)

cvars = np.array([r["cvar_oos"] for r in monthly_results.values()])
print(f"Average monthly CVaR:  {cvars.mean():.6f}")
print(f"Std dev across months: {cvars.std():.6f}")
print(f"Min / Max monthly CVaR: {cvars.min():.6f} / {cvars.max():.6f}")

Average monthly CVaR:  0.011122
Std dev across months: 0.004392
Min / Max monthly CVaR: 0.005334 / 0.018233


## Part 6 — Stability check

In [15]:
def check_stability(monthly_results, tol=0.02):
    months_sorted = sorted(monthly_results.keys())
    max_moves = []
    for i in range(1, len(months_sorted)):
        w_prev = monthly_results[months_sorted[i-1]]["weights"]
        w_curr = monthly_results[months_sorted[i]]["weights"]
        max_moves.append(np.max(np.abs(w_curr - w_prev)))
    return months_sorted[1:], np.array(max_moves)

months, max_moves = check_stability(monthly_results)
for mo, move in zip(months, max_moves):
    flag = " <-- exceeds 2pp" if move > 0.02 else ""
    print(f"{mo}: max weight change = {move:.4f}{flag}")

(2026, 2): max weight change = 0.0695 <-- exceeds 2pp
(2026, 3): max weight change = 0.0397 <-- exceeds 2pp
(2026, 4): max weight change = 0.0682 <-- exceeds 2pp
(2026, 5): max weight change = 0.1461 <-- exceeds 2pp
(2026, 6): max weight change = 0.1016 <-- exceeds 2pp
(2026, 7): max weight change = 0.0972 <-- exceeds 2pp
(2026, 8): max weight change = 0.0674 <-- exceeds 2pp
(2026, 9): max weight change = 0.0474 <-- exceeds 2pp


How much weights move between consecutive months 